# LexGLUE Legal RAG System: Hukuki Belgelerin Yerel LLM ile İşlenmesi ve Sorgulanması

**Proje Amacı:** Hukuki belgelerin yerel LLM mimarisi ile işlenmesi ve sorgulanması için çözüm tasarımı.

**Dataset:** [LexGLUE - Legal NLP Benchmark Dataset (Kaggle)](https://www.kaggle.com/datasets/thedevastator/lexglue-legal-nlp-benchmark-dataset)

**Yaklaşım:** 3 aşamalı RAG pipeline karşılaştırması (Basic → Adaptive → Fusion) + 3 farklı LLM değerlendirmesi

---

## İçindekiler

| # | Bölüm | Açıklama |
|---|---|---|
| 1 | Veri Yükleme & EDA | 7 alt dataset'in yüklenmesi, keşifsel analiz |
| 2 | Ön İşleme & Chunking | Text temizleme, metadata şeması, chunk stratejisi |
| 3 | Embedding & Vektör DB | ChromaDB'ye indexleme, embedding pipeline |
| 4 | Basic RAG | Naive retrieval + generation pipeline |
| 5 | Adaptive RAG | Query complexity routing |
| 6 | RAG Fusion | Multi-query + Reciprocal Rank Fusion |
| 7 | LLM Karşılaştırması | SaulLM-7B vs Mistral 7B vs Llama 3 8B |
| 8 | Değerlendirme & Sonuçlar | Metrikler, karşılaştırma tablosu, ileri vizyon |

---
## 0. Ortam Kurulumu

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
import json
import ast
import re
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 200)

---
## 1. Veri Yükleme & Keşifsel Analiz (EDA)

LexGLUE benchmark dataset'i 7 alt hukuki NLP görevinden oluşur:

| Dataset | Açıklama | Hukuki Alan |
|---|---|---|
| **case_hold** | Case holdings — hangi hukuki yorum geçerli? | ABD İçtihat Hukuku |
| **ecthr_a** | Avrupa İnsan Hakları Mahkemesi — ihlal edilen madde(ler) | İnsan Hakları |
| **ecthr_b** | ECtHR — ihlal edilen madde(ler) (farklı granularity) | İnsan Hakları |
| **eurlex** | AB mevzuatı — EUROVOC sınıflandırması | AB Mevzuatı |
| **scotus** | ABD Yüksek Mahkemesi — konu kategorisi | ABD Anayasa Hukuku |
| **ledgar** | Sözleşme maddeleri — madde tipi sınıflandırması | Sözleşme Hukuku |
| **unfair_tos** | Hizmet şartları — haksız madde tespiti | Tüketici Hukuku |

### 1.1 Dataset İndirme

**Kaggle Dataset Linki:** https://www.kaggle.com/datasets/thedevastator/lexglue-legal-nlp-benchmark-dataset

In [33]:
DATA_DIR = "data/"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Data dizini: {DATA_DIR}")
print(f"Mevcut dosyalar: {os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else 'Boş'}")

Data dizini: data/
Mevcut dosyalar: ['unfair_tos_validation.csv', 'case_hold_validation.csv', 'scotus_test.csv', 'ecthr_b_test.csv', 'ecthr_a_train.csv', 'scotus_validation.csv', 'ledgar_validation.csv', 'eurlex_test.csv', 'ecthr_b_validation.csv', 'ecthr_a_test.csv', 'case_hold_train.csv', 'eurlex_train.csv', 'case_hold_test.csv', 'scotus_train.csv', 'unfair_tos_train.csv', 'ecthr_a_validation.csv', 'ledgar_test.csv', 'eurlex_validation.csv', 'ecthr_b_train.csv', 'unfair_tos_test.csv', 'ledgar_train.csv']


In [3]:
# 7 alt dataset tanımları
DATASETS = {
    "case_hold": {
        "domain": "case_law",
        "description": "ABD İçtihat Hukuku - Case Holdings",
        "text_col": "context",
        "label_col": "label",
        "multi_label": False,
        "has_endings": True
    },
    "ecthr_a": {
        "domain": "human_rights",
        "description": "AİHM - İhlal Edilen Maddeler (A)",
        "text_col": "text",
        "label_col": "labels",
        "multi_label": True,
        "has_endings": False
    },
    "ecthr_b": {
        "domain": "human_rights",
        "description": "AİHM - İhlal Edilen Maddeler (B)",
        "text_col": "text",
        "label_col": "labels",
        "multi_label": True,
        "has_endings": False
    },
    "eurlex": {
        "domain": "eu_legislation",
        "description": "AB Mevzuatı - EUROVOC Sınıflandırması",
        "text_col": "text",
        "label_col": "labels",
        "multi_label": True,
        "has_endings": False
    },
    "scotus": {
        "domain": "supreme_court",
        "description": "ABD Yüksek Mahkemesi - Konu Sınıflandırması",
        "text_col": "text",
        "label_col": "label",
        "multi_label": False,
        "has_endings": False
    },
    "ledgar": {
        "domain": "contracts",
        "description": "Sözleşme Maddeleri - Tip Sınıflandırması",
        "text_col": "text",
        "label_col": "label",
        "multi_label": False,
        "has_endings": False
    },
    "unfair_tos": {
        "domain": "consumer_law",
        "description": "Hizmet Şartları - Haksız Madde Tespiti",
        "text_col": "text",
        "label_col": "labels",
        "multi_label": True,
        "has_endings": False
    }
}

SPLITS = ["train", "test", "validation"]

print(f"{len(DATASETS)} dataset, {len(SPLITS)} split = {len(DATASETS) * len(SPLITS)} CSV dosyası")

7 dataset, 3 split = 21 CSV dosyası


In [4]:
# Tüm CSV'leri yükle
all_data = {}
load_summary = []

for ds_name, ds_config in DATASETS.items():
    for split in SPLITS:
        filename = f"{ds_name}_{split}.csv"
        filepath = os.path.join(DATA_DIR, filename)
        
        if os.path.exists(filepath):
            df = pd.read_csv(filepath)
            key = f"{ds_name}_{split}"
            all_data[key] = df
            load_summary.append({
                "dataset": ds_name,
                "split": split,
                "rows": len(df),
                "columns": list(df.columns),
                "domain": ds_config["domain"]
            })
            print(f" {filename}: {len(df):,} rows, columns={list(df.columns)}")
        else:
            print(f" {filename}: Dosya bulunamadı")

print(f"\n Toplam yüklenen: {len(all_data)} dosya")
print(f"Toplam kayıt: {sum(item['rows'] for item in load_summary):,}")

 case_hold_train.csv: 45,000 rows, columns=['context', 'endings', 'label']
 case_hold_test.csv: 3,600 rows, columns=['context', 'endings', 'label']
 case_hold_validation.csv: 3,900 rows, columns=['context', 'endings', 'label']
 ecthr_a_train.csv: 9,000 rows, columns=['text', 'labels']
 ecthr_a_test.csv: 1,000 rows, columns=['text', 'labels']
 ecthr_a_validation.csv: 1,000 rows, columns=['text', 'labels']
 ecthr_b_train.csv: 9,000 rows, columns=['text', 'labels']
 ecthr_b_test.csv: 1,000 rows, columns=['text', 'labels']
 ecthr_b_validation.csv: 1,000 rows, columns=['text', 'labels']
 eurlex_train.csv: 55,000 rows, columns=['text', 'labels']
 eurlex_test.csv: 5,000 rows, columns=['text', 'labels']
 eurlex_validation.csv: 5,000 rows, columns=['text', 'labels']
 scotus_train.csv: 5,000 rows, columns=['text', 'label']
 scotus_test.csv: 1,400 rows, columns=['text', 'label']
 scotus_validation.csv: 1,400 rows, columns=['text', 'label']
 ledgar_train.csv: 60,000 rows, columns=['text', 'label']

### 1.2 Dataset Boyutları & Dağılım Özeti

In [5]:
# Özet tablo
summary_df = pd.DataFrame(load_summary)

# Pivot: dataset x split → row counts
pivot = summary_df.pivot_table(
    index=["dataset", "domain"], 
    columns="split", 
    values="rows", 
    aggfunc="sum"
).fillna(0).astype(int)

pivot["TOTAL"] = pivot.sum(axis=1)
pivot = pivot.sort_values("TOTAL", ascending=False)

print("Dataset Boyutları (satır sayısı):")
print("=" * 70)
display(pivot) if hasattr(__builtins__, '__IPYTHON__') else print(pivot)
print(f"\n Toplam corpus boyutu: {pivot['TOTAL'].sum():,} kayıt")

Dataset Boyutları (satır sayısı):


,split,test,train,validation,TOTAL
dataset,domain,,,,
ledgar,contracts,10000,60000,10000,80000
eurlex,eu_legislation,5000,55000,5000,65000
case_hold,case_law,3600,45000,3900,52500
ecthr_a,human_rights,1000,9000,1000,11000
ecthr_b,human_rights,1000,9000,1000,11000
unfair_tos,consumer_law,1607,5532,2275,9414
scotus,supreme_court,1400,5000,1400,7800



 Toplam corpus boyutu: 236,714 kayıt


In [6]:
# Dataset boyutları visualizasyonu
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Dataset Boyutları (Split Bazında)", "Domain Dağılımı"),
    specs=[[{"type": "bar"}, {"type": "pie"}]]
)

colors = {"train": "#2196F3", "validation": "#FF9800", "test": "#4CAF50"}

for split in SPLITS:
    split_data = summary_df[summary_df["split"] == split]
    fig.add_trace(
        go.Bar(
            name=split,
            x=split_data["dataset"],
            y=split_data["rows"],
            marker_color=colors[split]
        ),
        row=1, col=1
    )

# Domain dağılımı
domain_counts = summary_df.groupby("domain")["rows"].sum()
fig.add_trace(
    go.Pie(
        labels=domain_counts.index,
        values=domain_counts.values,
        hole=0.4
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="LexGLUE Corpus Genel Bakış",
    height=450,
    barmode="stack",
    template="plotly_white"
)
fig.show()

### 1.3 Text Uzunluk Analizi

RAG sistemi için kritik: chunk stratejisini belirlemek için metin uzunluk dağılımını anlamamız gerekiyor.

In [7]:
# Her dataset için text uzunluk istatistikleri
text_stats = []

for ds_name, ds_config in DATASETS.items():
    text_col = ds_config["text_col"]
    
    for split in SPLITS:
        key = f"{ds_name}_{split}"
        if key not in all_data:
            continue
        
        df = all_data[key]
        if text_col not in df.columns:
            # case_hold uses 'context' column
            text_col_actual = "context" if "context" in df.columns else text_col
        else:
            text_col_actual = text_col
            
        if text_col_actual in df.columns:
            lengths = df[text_col_actual].astype(str).str.len()
            word_counts = df[text_col_actual].astype(str).str.split().str.len()
            
            text_stats.append({
                "dataset": ds_name,
                "split": split,
                "domain": ds_config["domain"],
                "mean_chars": lengths.mean(),
                "median_chars": lengths.median(),
                "max_chars": lengths.max(),
                "mean_words": word_counts.mean(),
                "median_words": word_counts.median(),
                "max_words": word_counts.max()
            })

stats_df = pd.DataFrame(text_stats)
print(" Text Uzunluk İstatistikleri (karakter):")
print("=" * 80)

# Dataset bazında ortalama
avg_stats = stats_df.groupby("dataset").agg({
    "mean_chars": "mean",
    "median_chars": "mean",
    "max_chars": "max",
    "mean_words": "mean",
    "median_words": "mean",
    "max_words": "max"
}).round(0).astype(int)

avg_stats.columns = ["Ort. Karakter", "Medyan Karakter", "Max Karakter", 
                     "Ort. Kelime", "Medyan Kelime", "Max Kelime"]
display(avg_stats) if hasattr(__builtins__, '__IPYTHON__') else print(avg_stats)

 Text Uzunluk İstatistikleri (karakter):


,Ort. Karakter,Medyan Karakter,Max Karakter,Ort. Kelime,Medyan Kelime,Max Kelime
dataset,,,,,,
case_hold,843,849,855,136,137,207
ecthr_a,10844,7419,211709,1771,1215,35377
ecthr_b,10844,7419,211709,1771,1215,35377
eurlex,9058,2911,1269472,1439,466,200749
ledgar,692,514,7803,111,83,1215
scotus,48001,38800,577927,7722,6280,89379
unfair_tos,183,150,2535,33,27,441


In [8]:
# Text uzunluk dağılımı box plot
length_data = []

for ds_name, ds_config in DATASETS.items():
    key = f"{ds_name}_train"  # Train split yeterli
    if key not in all_data:
        continue
    
    df = all_data[key]
    text_col = "context" if "context" in df.columns else ds_config["text_col"]
    
    if text_col in df.columns:
        word_counts = df[text_col].astype(str).str.split().str.len()
        sample = word_counts.sample(min(1000, len(word_counts)), random_state=42)
        for wc in sample:
            length_data.append({"dataset": ds_name, "word_count": wc})

length_df = pd.DataFrame(length_data)

fig = px.box(
    length_df, x="dataset", y="word_count",
    color="dataset",
    title="Text Uzunluk Dağılımı (Kelime Sayısı, Train Split)",
    labels={"word_count": "Kelime Sayısı", "dataset": "Dataset"}
)
fig.update_layout(height=450, template="plotly_white", showlegend=False)
fig.show()

print("\n Chunking stratejisi için önemli bulgular:")
print("  - Kısa metinler (ledgar, unfair_tos): Chunk'a gerek yok, doğrudan embed")
print("  - Orta metinler (case_hold, scotus): 512 token chunk yeterli")
print("  - Uzun metinler (ecthr_a/b, eurlex): Overlap ile chunking gerekli")


 Chunking stratejisi için önemli bulgular:
  - Kısa metinler (ledgar, unfair_tos): Chunk'a gerek yok, doğrudan embed
  - Orta metinler (case_hold, scotus): 512 token chunk yeterli
  - Uzun metinler (ecthr_a/b, eurlex): Overlap ile chunking gerekli


### 1.4 Label Dağılım Analizi

In [9]:
def to_label_list(x):
    """Return labels as a Python list regardless of source format."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    # If already a list/tuple/set -> return list
    if isinstance(x, (list, tuple, set)):
        return list(x)

    # If it's a numpy scalar etc -> stringify
    s = str(x).strip()

    # Normalize common empties
    if s in {"", "[]", "nan", "None", "null"}:
        return []

    # 1) Try JSON first (handles ["a", "b"] or [1,2])
    try:
        return json.loads(s)
    except Exception:
        pass

    # 2) Try python literal_eval (handles [4], ['a'])
    try:
        return ast.literal_eval(s)
    except Exception:
        pass

    # 3) Fallback: extract integers (handles "[4 5]" or "4,5" or "4|5")
    nums = re.findall(r"-?\d+", s)
    if nums:
        return [int(n) for n in nums]

    # 4) Last resort: treat as single label string
    return [s]
for ds_name, ds_config in DATASETS.items():
    key = f"{ds_name}_train"
    if key not in all_data:
        continue
    
    df = all_data[key]
    label_col = ds_config["label_col"]
    
    if label_col not in df.columns:
        print(f" {ds_name}: '{label_col}' kolonu bulunamadı. Mevcut: {list(df.columns)}")
        continue
    
    print(f"\n{'='*60}")
    print(f" {ds_name.upper()} — {ds_config['description']}")
    print(f"{'='*60}")
    
    if ds_config["multi_label"]:
        labels_as_lists = df[label_col].apply(to_label_list)
        labels_flat = labels_as_lists.explode()

        # explode sonrası boşları temizle
        labels_flat = labels_flat.dropna()
        labels_flat = labels_flat[labels_flat != ""]  # string label varsa
        
        print(f"  Benzersiz label sayısı: {labels_flat.nunique()}")
        print(f"  Ortalama label/kayıt: {labels_as_lists.apply(len).mean():.2f}")
        print(f"  En sık 10 label:")
        print(labels_flat.value_counts().head(10).to_string())
        
    else:
        print(f"  Benzersiz label sayısı: {df[label_col].nunique()}")
        print(f"  Label dağılımı:")
        print(df[label_col].value_counts().head(15).to_string())


 CASE_HOLD — ABD İçtihat Hukuku - Case Holdings
  Benzersiz label sayısı: 5
  Label dağılımı:
label
0    9108
2    9066
4    9008
3    8944
1    8874

 ECTHR_A — AİHM - İhlal Edilen Maddeler (A)
  Benzersiz label sayısı: 10
  Ortalama label/kayıt: 1.18
  En sık 10 label:
labels
3    4704
9    1421
2    1368
1    1349
4     710
0     505
6     291
8     141
7     110
5      41

 ECTHR_B — AİHM - İhlal Edilen Maddeler (B)
  Benzersiz label sayısı: 10
  Ortalama label/kayıt: 1.46
  En sık 10 label:
labels
3    5437
1    1740
2    1623
9    1558
4    1056
0     623
8     444
6     441
7     162
5      81

 EURLEX — AB Mevzuatı - EUROVOC Sınıflandırması
  Benzersiz label sayısı: 100
  Ortalama label/kayıt: 4.42
  En sık 10 label:
labels
96    17781
97    14803
91    13702
68    11982
20    11277
21    10804
22     7739
61     7470
66     7072
4      6335

 SCOTUS — ABD Yüksek Mahkemesi - Konu Sınıflandırması
  Benzersiz label sayısı: 13
  Label dağılımı:
label
7     1043
0     1011
1      

In [10]:
# Örnek kayıtlar — her dataset'ten 1 örnek
print(" Örnek Kayıtlar (Her Dataset'ten 1 Örnek)")
print("=" * 80)

for ds_name, ds_config in DATASETS.items():
    key = f"{ds_name}_train"
    if key not in all_data:
        continue
    
    df = all_data[key]
    sample = df.iloc[0]
    text_col = "context" if "context" in df.columns else ds_config["text_col"]
    
    print(f"\n--- {ds_name.upper()} ({ds_config['domain']}) ---")
    if text_col in df.columns:
        text_preview = str(sample[text_col])[:300] + "..." if len(str(sample[text_col])) > 300 else str(sample[text_col])
        print(f"  Text: {text_preview}")
    
    label_col = ds_config["label_col"]
    if label_col in df.columns:
        print(f"  Label: {sample[label_col]}")
    
    if ds_config["has_endings"] and "endings" in df.columns:
        print(f"  Endings: {str(sample['endings'])[:200]}...")

 Örnek Kayıtlar (Her Dataset'ten 1 Örnek)

--- CASE_HOLD (case_law) ---
  Text: Drapeau’s cohorts, the cohort would be a “victim” of making the bomb. Further, firebombs are inherently dangerous. There is no peaceful purpose for making a bomb. Felony offenses that involve explosives qualify as “violent crimes” for purposes of enhancing the sentences of career offenders. See 18 U...
  Label: 0
  Endings: ['holding that possession of a pipe bomb is a crime of violence for purposes of 18 usc  3142f1'
 'holding that bank robbery by force and violence or intimidation under 18 usc  2113a is a crime of viol...

--- ECTHR_A (human_rights) ---
  Text: ['11.  At the beginning of the events relevant to the application, K. had a daughter, P., and a son, M., born in 1986 and 1988 respectively. P.’s father is X and M.’s father is V. From March to May 1989 K. was voluntarily hospitalised for about three months, having been diagnosed as suffering from s...
  Label: [4]

--- ECTHR_B (human_rights) ---
 

### 1.5 EDA Özeti & Chunking Stratejisi Kararı

**Bulgular:**

| Dataset | Metin Uzunluğu | Chunking Stratejisi |
|---|---|---|
| ledgar | Kısa (~100 kelime) | Chunk yok — doğrudan embed |
| unfair_tos | Kısa (~50 kelime) | Chunk yok — doğrudan embed |
| case_hold | Orta (~200 kelime) | Chunk yok / tek chunk |
| scotus | Uzun (~2000+ kelime) | RecursiveCharacterTextSplitter, 512 token, 50 overlap |
| ecthr_a/b | Uzun (~2000+ kelime) | RecursiveCharacterTextSplitter, 512 token, 50 overlap |
| eurlex | Uzun (~1000+ kelime) | RecursiveCharacterTextSplitter, 512 token, 50 overlap |

**Kararlar:**
1. Tüm 7 dataset, tüm split'ler corpusa dahil
2. Metadata şemasında `source_dataset`, `split`, `label`, `domain`, `doc_id` tutulacak
3. Uzun metinler chunk'lanacak, kısa metinler olduğu gibi embed edilecek
4. Threshold: 512 kelimeden uzun metinler chunk'lanır

---
## 2. Ön İşleme & Chunking

Bu bölümde:
1. Text temizleme (whitespace normalization, encoding fix)
2. Unified document format oluşturma
3. Metadata şeması uygulama
4. Adaptive chunking (kısa metinler → doğrudan, uzun metinler → chunk)

In [11]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_text(text: str) -> str:
    """Hukuki metin temizleme."""
    if not isinstance(text, str) or pd.isna(text):
        return ""
    
    # Çoklu boşlukları teke indir
    text = re.sub(r'\s+', ' ', text)
    # Satır başı/sonu temizle
    text = text.strip()
    # Encoding artifacts
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    
    return text


def prepare_documents(
    all_data: dict, 
    datasets_config: dict,
    max_per_dataset: int = 5000,
    splits: list = None
) -> list[dict]:
    """
    Tüm dataset'lerden unified document listesi oluştur.
    
    Args:
        max_per_dataset: Her dataset başına max kayıt (RAM/zaman optimizasyonu)
        splits: Hangi split'ler alınacak (default: sadece train)
    
    Not: Production'da max_per_dataset=None ve splits=["train","test","validation"] kullanılır.
         Demo için temsili örneklem yeterlidir.
    """
    if splits is None:
        splits = ["train"]
    
    documents = []
    doc_counter = 0
    
    for ds_name, ds_config in datasets_config.items():
        ds_doc_count = 0
        
        for split in splits:
            key = f"{ds_name}_{split}"
            if key not in all_data:
                continue
            
            df = all_data[key]
            text_col = "context" if "context" in df.columns else ds_config["text_col"]
            label_col = ds_config["label_col"]
            
            if text_col not in df.columns:
                continue
            
            # Sampling: dataset başına limit
            remaining = max_per_dataset - ds_doc_count
            if remaining <= 0:
                break
            df_sample = df.sample(n=min(len(df), remaining), random_state=42) if len(df) > remaining else df
            
            for idx, row in df_sample.iterrows():
                text = clean_text(str(row[text_col]))
                if len(text) < 20:
                    continue
                
                label_value = str(row.get(label_col, "")) if label_col in df.columns else ""
                
                endings_value = ""
                if ds_config["has_endings"] and "endings" in df.columns:
                    endings_value = str(row["endings"])
                    text = f"{text}\n\nPossible Holdings: {endings_value}"
                
                metadata = {
                    "source_dataset": ds_name,
                    "split": split,
                    "domain": ds_config["domain"],
                    "label": label_value,
                    "multi_label": ds_config["multi_label"],
                    "doc_id": f"{ds_name}_{split}_{doc_counter:06d}",
                    "description": ds_config["description"]
                }
                
                documents.append({"text": text, "metadata": metadata})
                doc_counter += 1
                ds_doc_count += 1
    
    return documents


# ============================================
# SAMPLING STRATEJİSİ
# ============================================
# Her dataset'ten max 5,000 kayıt, sadece train split
# Sonuç: ~35K belge → chunking sonrası ~50-80K chunk → ~10-15 dk
# Production'da: max_per_dataset parametresini kaldır, tüm split'leri al

MAX_PER_DATASET = 500
INDEX_SPLITS = ["train"]

print(f"⚙️ Sampling: max {MAX_PER_DATASET} kayıt/dataset, splits={INDEX_SPLITS}")
print(f"   Production'da tüm corpus indexlenebilir.\n")

documents = prepare_documents(
    all_data, DATASETS, 
    max_per_dataset=MAX_PER_DATASET, 
    splits=INDEX_SPLITS
)

total_full_corpus = sum(item['rows'] for item in load_summary)
print(f"📄 Örneklem: {len(documents):,} belge (full corpus: {total_full_corpus:,})")
print(f"   Örneklem oranı: {len(documents)/total_full_corpus*100:.1f}%")
print(f"\n📊 Dataset bazında dağılım:")
ds_counts = Counter(d['metadata']['source_dataset'] for d in documents)
for ds, count in sorted(ds_counts.items(), key=lambda x: -x[1]):
    print(f"  {ds}: {count:,}")

⚙️ Sampling: max 500 kayıt/dataset, splits=['train']
   Production'da tüm corpus indexlenebilir.

📄 Örneklem: 3,500 belge (full corpus: 236,714)
   Örneklem oranı: 1.5%

📊 Dataset bazında dağılım:
  case_hold: 500
  ecthr_a: 500
  ecthr_b: 500
  eurlex: 500
  scotus: 500
  ledgar: 500
  unfair_tos: 500


In [12]:
# Adaptive Chunking
CHUNK_THRESHOLD = 512  # kelime
CHUNK_SIZE = 1000      # karakter
CHUNK_OVERLAP = 150    # karakter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def adaptive_chunk(documents: list[dict]) -> list[dict]:
    """
    Kısa metinler → olduğu gibi
    Uzun metinler → RecursiveCharacterTextSplitter ile chunk'la
    """
    chunked_docs = []
    stats = {"kept_as_is": 0, "chunked": 0, "total_chunks": 0}
    
    for doc in documents:
        word_count = len(doc["text"].split())
        
        if word_count <= CHUNK_THRESHOLD:
            # Kısa metin — doğrudan ekle
            chunked_docs.append({
                "text": doc["text"],
                "metadata": {**doc["metadata"], "chunk_id": 0, "total_chunks": 1}
            })
            stats["kept_as_is"] += 1
        else:
            # Uzun metin — chunk'la
            chunks = text_splitter.split_text(doc["text"])
            for i, chunk in enumerate(chunks):
                chunked_docs.append({
                    "text": chunk,
                    "metadata": {
                        **doc["metadata"],
                        "chunk_id": i,
                        "total_chunks": len(chunks),
                        "doc_id": f"{doc['metadata']['doc_id']}_chunk{i:03d}"
                    }
                })
            stats["chunked"] += 1
            stats["total_chunks"] += len(chunks)
    
    return chunked_docs, stats


chunked_documents, chunk_stats = adaptive_chunk(documents)

print(f" Chunking Sonuçları:")
print(f"  Orijinal belge sayısı: {len(documents):,}")
print(f"  Chunk'lanmadan geçen: {chunk_stats['kept_as_is']:,}")
print(f"  Chunk'lanan belge: {chunk_stats['chunked']:,}")
print(f"  Oluşan toplam chunk: {chunk_stats['total_chunks']:,}")
print(f"  \n  Vektör DB'ye girecek toplam kayıt: {len(chunked_documents):,}")

 Chunking Sonuçları:
  Orijinal belge sayısı: 3,500
  Chunk'lanmadan geçen: 2,106
  Chunk'lanan belge: 1,394
  Oluşan toplam chunk: 35,864
  
  Vektör DB'ye girecek toplam kayıt: 37,970


In [13]:
# Chunk uzunluk dağılımı
chunk_lengths = [len(d["text"].split()) for d in chunked_documents]

fig = px.histogram(
    x=chunk_lengths,
    nbins=50,
    title="Chunk Uzunluk Dağılımı (Kelime Sayısı)",
    labels={"x": "Kelime Sayısı", "y": "Frekans"},
    color_discrete_sequence=["#2196F3"]
)
fig.add_vline(x=np.median(chunk_lengths), line_dash="dash", line_color="red",
              annotation_text=f"Medyan: {np.median(chunk_lengths):.0f}")
fig.update_layout(height=400, template="plotly_white")
fig.show()

print(f"\n Chunk İstatistikleri:")
print(f"  Min: {min(chunk_lengths)} kelime")
print(f"  Max: {max(chunk_lengths)} kelime")
print(f"  Ortalama: {np.mean(chunk_lengths):.0f} kelime")
print(f"  Medyan: {np.median(chunk_lengths):.0f} kelime")


 Chunk İstatistikleri:
  Min: 2 kelime
  Max: 510 kelime
  Ortalama: 146 kelime
  Medyan: 150 kelime


---
## 3. Embedding & Vektör Veritabanı (ChromaDB)

**Embedding Model:** `all-MiniLM-L6-v2` (384 boyut, hızlı, RAG için optimize)

**Vektör DB:** ChromaDB (persistent, metadata filtering destekli)

In [14]:
import chromadb
from chromadb.utils import embedding_functions

# Embedding fonksiyonu
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

# ChromaDB client (persistent)
CHROMA_DIR = "./chroma_db"
client = chromadb.PersistentClient(path=CHROMA_DIR)

# Collection oluştur (varsa sil ve yeniden oluştur)
COLLECTION_NAME = "lexglue_legal_corpus"

try:
    client.delete_collection(COLLECTION_NAME)
except:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

print(f" ChromaDB collection oluşturuldu: '{COLLECTION_NAME}'")
print(f" Embedding model: {EMBEDDING_MODEL} (384 dim)")
print(f" Distance metric: cosine")

 ChromaDB collection oluşturuldu: 'lexglue_legal_corpus'
 Embedding model: all-MiniLM-L6-v2 (384 dim)
 Distance metric: cosine


In [15]:
# Batch indexleme
BATCH_SIZE = 500

total = len(chunked_documents)
print(f"{total:,} chunk indexleniyor (batch size={BATCH_SIZE})...")

for i in range(0, total, BATCH_SIZE):
    batch = chunked_documents[i:i+BATCH_SIZE]
    
    ids = [d["metadata"]["doc_id"] for d in batch]
    texts = [d["text"] for d in batch]
    metadatas = []
    
    for d in batch:
        # ChromaDB metadata: sadece string, int, float, bool kabul eder
        meta = {
            "source_dataset": d["metadata"]["source_dataset"],
            "split": d["metadata"]["split"],
            "domain": d["metadata"]["domain"],
            "label": str(d["metadata"]["label"]),
            "chunk_id": d["metadata"]["chunk_id"],
            "total_chunks": d["metadata"]["total_chunks"]
        }
        metadatas.append(meta)
    
    collection.add(
        ids=ids,
        documents=texts,
        metadatas=metadatas
    )
    
    if (i + BATCH_SIZE) % 5000 == 0 or i + BATCH_SIZE >= total:
        print(f" {min(i+BATCH_SIZE, total):,}/{total:,} indexlendi ({(min(i+BATCH_SIZE, total))/total*100:.1f}%)")

print(f"\n İndexleme tamamlandı! Collection boyutu: {collection.count():,}")

37,970 chunk indexleniyor (batch size=500)...
 5,000/37,970 indexlendi (13.2%)
 10,000/37,970 indexlendi (26.3%)
 15,000/37,970 indexlendi (39.5%)
 20,000/37,970 indexlendi (52.7%)
 25,000/37,970 indexlendi (65.8%)
 30,000/37,970 indexlendi (79.0%)
 35,000/37,970 indexlendi (92.2%)
 37,970/37,970 indexlendi (100.0%)

 İndexleme tamamlandı! Collection boyutu: 37,970


In [16]:
# Hızlı doğrulama — test sorgusu
test_queries = [
    "right to fair trial article 6",
    "breach of contract termination clause",
    "freedom of expression limitations"
]

for query in test_queries:
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    
    print(f"\n Query: '{query}'")
    for j, (doc, meta, dist) in enumerate(zip(
        results["documents"][0], 
        results["metadatas"][0],
        results["distances"][0]
    )):
        print(f"  [{j+1}] Score: {1-dist:.4f} | {meta['source_dataset']} ({meta['domain']})")
        print(f"      {doc[:150]}...")


 Query: 'right to fair trial article 6'
  [1] Score: 0.6660 | scotus (supreme_court)
      . The Framers did not declare in the Sixth Amendment that a defendant is entitled to a 'fair trial,' nor that he is entitled to counsel on the conditi...
  [2] Score: 0.6504 | ecthr_a (human_rights)
      . It held that an ordinary reader would have definitely understood the applicant’s statement to refer to Dr Boffa’s wish to build there personally and...
  [3] Score: 0.6504 | ecthr_b (human_rights)
      . It held that an ordinary reader would have definitely understood the applicant’s statement to refer to Dr Boffa’s wish to build there personally and...

 Query: 'breach of contract termination clause'
  [1] Score: 0.6554 | ledgar (contracts)
      If Executive breaches any of the provisions contained in Paragraphs 5, 6 or 7 above, Company shall have the right to immediately terminate all payment...
  [2] Score: 0.6343 | ledgar (contracts)
      Notwithstanding anything to the contrary contai

In [17]:
# Metadata-based filtered search test
print(" Metadata Filtered Search Testi:")
print("=" * 60)

# Sadece ECtHR kararlarında ara
results = collection.query(
    query_texts=["right to privacy surveillance"],
    n_results=3,
    where={"domain": "human_rights"}
)
print("\n Domain filtresi: human_rights")
for meta in results["metadatas"][0]:
    print(f"   → {meta['source_dataset']} | {meta['domain']}")

# Sadece sözleşme maddelerinde ara
results = collection.query(
    query_texts=["indemnification liability"],
    n_results=3,
    where={"source_dataset": "ledgar"}
)
print("\n Dataset filtresi: ledgar")
for meta in results["metadatas"][0]:
    print(f"   → {meta['source_dataset']} | {meta['domain']}")

print("\n Metadata filtering çalışıyor!")

 Metadata Filtered Search Testi:

 Domain filtresi: human_rights
   → ecthr_a | human_rights
   → ecthr_b | human_rights
   → ecthr_a | human_rights

 Dataset filtresi: ledgar
   → ledgar | contracts
   → ledgar | contracts
   → ledgar | contracts

 Metadata filtering çalışıyor!


---
## 4. Basic RAG (Naive RAG)

**Pipeline:** `Query → Embed → Top-K Retrieval → LLM Prompt → Answer`

En basit RAG yaklaşımı. Tek sorgu, tek retrieval, tek LLM çağrısı.

In [21]:
import requests
from typing import Optional


LLM_BASE_URL = "http://192.168.1.15:1234"
LLM_MODEL = "llama3-8b-lawyer-v2"


def _extract_text_from_json(j: dict) -> Optional[str]:
    """
    Farklı server'ların döndürebileceği olası şemalardan metni çek.
    """
    if not isinstance(j, dict):
        return None

    # Ollama /api/generate
    if "response" in j and isinstance(j["response"], str):
        return j["response"]

    # Bazı generate şemaları
    for k in ["output", "completion", "text", "content", "generated_text"]:
        if k in j and isinstance(j[k], str):
            return j[k]

    # OpenAI-compatible chat completions
    try:
        return j["choices"][0]["message"]["content"]
    except Exception:
        pass

    # OpenAI-compatible (bazı impl. chat yerine text döndürür)
    try:
        return j["choices"][0]["text"]
    except Exception:
        pass

    return None


def call_llm(prompt: str, model: Optional[str] = None, temperature: float = 0.1, debug: bool = False) -> str:
    model = model or LLM_MODEL
    headers = {"Content-Type": "application/json"}

    try:
        r = requests.post(
            f"{LLM_BASE_URL}/v1/chat/completions",
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": temperature,
            },
            headers=headers,
            timeout=120,
        )
        if r.ok:
            j = r.json()
            text = _extract_text_from_json(j)
            if text is not None:
                return text
            if debug:
                return f"[DEBUG /v1/chat/completions JSON]\n{j}"
        else:
            if debug:
                return f"[DEBUG /v1/chat/completions HTTP {r.status_code}]\n{r.text}"
    except Exception as e:
        if debug:
            return f"[DEBUG /v1/chat/completions EXC]\n{repr(e)}"


    return "[LLM bağlantısı kurulamadı veya beklenmeyen cevap şeması geldi. debug=True ile deneyip çıktıyı paylaş.]"


# Test
test_response = call_llm("Say 'Legal RAG system is ready' in one sentence.", debug=True)
print(test_response[:300])

Legal RAG system is ready


In [ ]:
# Her dataset için text uzunluk istatistikleri
text_stats = []

for ds_name, ds_config in DATASETS.items():
    text_col = ds_config["text_col"]
    
    for split in SPLITS:
        key = f"{ds_name}_{split}"
        if key not in all_data:
            continue
        
        df = all_data[key]
        if text_col not in df.columns:
            # case_hold uses 'context' column
            text_col_actual = "context" if "context" in df.columns else text_col
        else:
            text_col_actual = text_col
            
        if text_col_actual in df.columns:
            lengths = df[text_col_actual].astype(str).str.len()
            word_counts = df[text_col_actual].astype(str).str.split().str.len()
            
            text_stats.append({
                "dataset": ds_name,
                "split": split,
                "domain": ds_config["domain"],
                "mean_chars": lengths.mean(),
                "median_chars": lengths.median(),
                "max_chars": lengths.max(),
                "mean_words": word_counts.mean(),
                "median_words": word_counts.median(),
                "max_words": word_counts.max()
            })

stats_df = pd.DataFrame(text_stats)
print(" Text Uzunluk İstatistikleri (karakter):")
print("=" * 80)

# Dataset bazında ortalama
avg_stats = stats_df.groupby("dataset").agg({
    "mean_chars": "mean",
    "median_chars": "mean",
    "max_chars": "max",
    "mean_words": "mean",
    "median_words": "mean",
    "max_words": "max"
}).round(0).astype(int)

avg_stats.columns = ["Ort. Karakter", "Medyan Karakter", "Max Karakter", 
                     "Ort. Kelime", "Medyan Kelime", "Max Kelime"]
display(avg_stats) if hasattr(__builtins__, '__IPYTHON__') else print(avg_stats)

 Text Uzunluk İstatistikleri (karakter):


,Ort. Karakter,Medyan Karakter,Max Karakter,Ort. Kelime,Medyan Kelime,Max Kelime
dataset,,,,,,
case_hold,843,849,855,136,137,207
ecthr_a,10844,7419,211709,1771,1215,35377
ecthr_b,10844,7419,211709,1771,1215,35377
eurlex,9058,2911,1269472,1439,466,200749
ledgar,692,514,7803,111,83,1215
scotus,48001,38800,577927,7722,6280,89379
unfair_tos,183,150,2535,33,27,441


In [22]:
# ============================================
# BASIC RAG PIPELINE
# ============================================

BASIC_RAG_PROMPT = """You are a legal assistant specialized in analyzing legal documents. 
Answer the question based ONLY on the provided context. If the context doesn't contain 
enough information, say so clearly. Always cite which source documents support your answer.

CONTEXT:
{context}

QUESTION: {question}

ANSWER (cite sources):"""


def basic_rag(
    question: str, 
    collection, 
    n_results: int = 5,
    model: str = None,
    domain_filter: str = None
) -> dict:
    """
    Basic (Naive) RAG Pipeline.
    
    Steps:
    1. Query'yi embed et
    2. Top-K retrieval (opsiyonel domain filtresi)
    3. Context oluştur
    4. LLM'e gönder
    5. Cevap + kaynakları döndür
    """
    import time
    start_time = time.time()
    
    # Step 1-2: Retrieval
    where_filter = {"domain": domain_filter} if domain_filter else None
    
    results = collection.query(
        query_texts=[question],
        n_results=n_results,
        where=where_filter
    )
    
    retrieval_time = time.time() - start_time
    
    # Step 3: Context oluştur
    context_parts = []
    sources = []
    
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        similarity = 1 - dist
        source_info = f"[Source {i+1}: {meta['source_dataset']} | {meta['domain']} | similarity={similarity:.3f}]"
        context_parts.append(f"{source_info}\n{doc}")
        sources.append({
            "rank": i + 1,
            "dataset": meta["source_dataset"],
            "domain": meta["domain"],
            "similarity": similarity,
            "text_preview": doc[:200]
        })
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 4: LLM çağrısı
    prompt = BASIC_RAG_PROMPT.format(context=context, question=question)
    
    llm_start = time.time()
    answer = call_llm(prompt, model=model)
    llm_time = time.time() - llm_start
    
    total_time = time.time() - start_time
    
    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "method": "basic_rag",
        "model": model or LLM_MODEL,
        "metrics": {
            "retrieval_time": retrieval_time,
            "llm_time": llm_time,
            "total_time": total_time,
            "n_sources": len(sources),
            "avg_similarity": np.mean([s["similarity"] for s in sources])
        }
    }


print("Basic RAG pipeline hazır.")

Basic RAG pipeline hazır.


In [23]:
# Basic RAG test
test_questions = [
    "What constitutes a violation of the right to fair trial under Article 6?",
    "Can an employer terminate a contract without notice for breach of confidentiality?",
    "What are the grounds for finding unfair terms in consumer service agreements?"
]

print("Basic RAG Test Sonuçları")
print("=" * 80)

basic_results = []
for q in test_questions:
    result = basic_rag(q, collection)
    basic_results.append(result)
    
    print(f"\n Soru: {q}")
    print(f" Cevap: {result['answer'][:300]}...")
    print(f" Süre: {result['metrics']['total_time']:.2f}s (retrieval: {result['metrics']['retrieval_time']:.3f}s)")
    print(f" Kaynaklar ({result['metrics']['n_sources']}):")
    for s in result["sources"][:3]:
        print(f"   [{s['rank']}] {s['dataset']} ({s['domain']}) — sim: {s['similarity']:.3f}")
    print("-" * 80)

Basic RAG Test Sonuçları

 Soru: What constitutes a violation of the right to fair trial under Article 6?
 Cevap: Under Article 6 of the European Convention on Human Rights, the right to a fair trial is generally understood to encompass several key elements. These include the right to be presumed innocent until proven guilty, the right to be informed of the charges and evidence against oneself, the right to exa...
 Süre: 12.35s (retrieval: 0.009s)
 Kaynaklar (5):
   [1] scotus (supreme_court) — sim: 0.665
   [2] ecthr_a (human_rights) — sim: 0.656
   [3] ecthr_b (human_rights) — sim: 0.656
--------------------------------------------------------------------------------

 Soru: Can an employer terminate a contract without notice for breach of confidentiality?
 Cevap: Under the terms of the agreement, the employer can terminate the contract without notice for breach of confidentiality. The agreement explicitly states that all agreements with the employee and obligations of the employee r

---
## 5. Adaptive RAG

**Pipeline:**
```
Query → Complexity Classifier → Route:
  ├─ SIMPLE    → Direct LLM (no retrieval)
  ├─ MODERATE  → Standard RAG (single retrieval)
  └─ COMPLEX   → Decompose → Multi-step RAG → Synthesize
```

Sorgunun karmaşıklığına göre farklı stratejiler uygulayan akıllı yönlendirme.

In [24]:
# ============================================
# ADAPTIVE RAG PIPELINE
# ============================================

CLASSIFY_PROMPT = """Classify the following legal question into one of three complexity levels.
Respond with ONLY one word: SIMPLE, MODERATE, or COMPLEX.

SIMPLE: Basic legal definitions, straightforward factual questions
MODERATE: Questions requiring analysis of specific legal provisions or case law
COMPLEX: Multi-faceted questions requiring cross-referencing multiple legal domains or comparative analysis

Question: {question}

Complexity:"""

DECOMPOSE_PROMPT = """Break down this complex legal question into 2-3 simpler sub-questions 
that can be answered independently and then synthesized.

Question: {question}

Return ONLY the sub-questions, one per line, numbered:
1.
2.
3."""

SYNTHESIZE_PROMPT = """You are a legal assistant. Synthesize the following sub-answers into 
a comprehensive response to the original question. Cite sources where applicable.

ORIGINAL QUESTION: {question}

SUB-ANSWERS:
{sub_answers}

SYNTHESIZED ANSWER:"""


def classify_complexity(question: str, model: str = None) -> str:
    """Sorunun karmaşıklık seviyesini belirle."""
    response = call_llm(CLASSIFY_PROMPT.format(question=question), model=model)
    response = response.strip().upper()
    
    # Parse
    for level in ["COMPLEX", "MODERATE", "SIMPLE"]:
        if level in response:
            return level
    return "MODERATE"  # default


def decompose_question(question: str, model: str = None) -> list[str]:
    """Karmaşık soruyu alt sorulara böl."""
    response = call_llm(DECOMPOSE_PROMPT.format(question=question), model=model)
    
    # Parse numbered lines
    sub_questions = []
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            # Remove numbering
            cleaned = re.sub(r'^[\d\.\-\)]+\s*', '', line).strip()
            if cleaned:
                sub_questions.append(cleaned)
    
    return sub_questions if sub_questions else [question]  # fallback


def adaptive_rag(
    question: str,
    collection,
    n_results: int = 5,
    model: str = None
) -> dict:
    """
    Adaptive RAG Pipeline.
    Sorgunun karmaşıklığına göre strateji seçer.
    """
    import time
    start_time = time.time()
    
    # Step 1: Complexity classification
    complexity = classify_complexity(question, model)
    
    all_sources = []
    llm_calls = 1  # classification call
    
    if complexity == "SIMPLE":
        # Direct LLM — minimal retrieval
        result = basic_rag(question, collection, n_results=2, model=model)
        answer = result["answer"]
        all_sources = result["sources"]
        llm_calls += 1
        
    elif complexity == "MODERATE":
        # Standard RAG
        result = basic_rag(question, collection, n_results=n_results, model=model)
        answer = result["answer"]
        all_sources = result["sources"]
        llm_calls += 1
        
    else:  # COMPLEX
        # Decompose → Multi-step RAG → Synthesize
        sub_questions = decompose_question(question, model)
        llm_calls += 1
        
        sub_answers = []
        for sq in sub_questions:
            sub_result = basic_rag(sq, collection, n_results=3, model=model)
            sub_answers.append(f"Q: {sq}\nA: {sub_result['answer']}")
            all_sources.extend(sub_result["sources"])
            llm_calls += 1
        
        # Synthesize
        synthesis_prompt = SYNTHESIZE_PROMPT.format(
            question=question,
            sub_answers="\n\n".join(sub_answers)
        )
        answer = call_llm(synthesis_prompt, model=model)
        llm_calls += 1
        
        # Deduplicate sources
        seen = set()
        unique_sources = []
        for s in all_sources:
            key = s["text_preview"][:100]
            if key not in seen:
                seen.add(key)
                unique_sources.append(s)
        all_sources = unique_sources
    
    total_time = time.time() - start_time
    
    return {
        "question": question,
        "answer": answer,
        "sources": all_sources,
        "method": "adaptive_rag",
        "complexity": complexity,
        "model": model or LLM_MODEL,
        "metrics": {
            "total_time": total_time,
            "llm_calls": llm_calls,
            "n_sources": len(all_sources),
            "avg_similarity": np.mean([s["similarity"] for s in all_sources]) if all_sources else 0
        }
    }


print("Adaptive RAG pipeline hazır.")

Adaptive RAG pipeline hazır.


In [25]:
# Adaptive RAG test
adaptive_test_questions = [
    # SIMPLE
    "What is the definition of habeas corpus?",
    # MODERATE
    "Under what conditions can Article 8 of ECHR be violated by state surveillance?",
    # COMPLEX
    "Compare how US case law and European human rights law treat the balance between freedom of expression and privacy rights, citing specific provisions."
]

print(" Adaptive RAG Test Sonuçları")
print("=" * 80)

adaptive_results = []
for q in adaptive_test_questions:
    result = adaptive_rag(q, collection)
    adaptive_results.append(result)
    
    print(f"\n Soru: {q}")
    print(f" Complexity: {result['complexity']}")
    print(f" Cevap: {result['answer'][:300]}...")
    print(f" Süre: {result['metrics']['total_time']:.2f}s | LLM calls: {result['metrics']['llm_calls']}")
    print(f" Kaynaklar: {result['metrics']['n_sources']}")
    print("-" * 80)

 Adaptive RAG Test Sonuçları

 Soru: What is the definition of habeas corpus?
 Complexity: SIMPLE
 Cevap: Habeas corpus is a Latin term meaning "you have the body." It refers to a legal procedure that allows a person who is being detained or imprisoned to challenge the legality of their detention. The writ of habeas corpus requires the custodian of the person (such as a prison warden) to produce the pri...
 Süre: 9.86s | LLM calls: 2
 Kaynaklar: 2
--------------------------------------------------------------------------------

 Soru: Under what conditions can Article 8 of ECHR be violated by state surveillance?
 Complexity: MODERATE
 Cevap: Under Article 8 of the European Convention on Human Rights, state surveillance may violate an individual's right to private and family life if it is not justified under one or more of the exceptions. The key factors are: 1) necessity; 2) proportionality; and 3) lawfulness. Surveillance must be neces...
 Süre: 13.09s | LLM calls: 2
 Kaynaklar: 5
---

---
## 6. RAG Fusion

**Pipeline:**
```
Query → Generate N variant queries →
  Each variant → Separate retrieval →
    Reciprocal Rank Fusion (RRF) → Re-ranked results → LLM → Answer
```

Tek query'nin kaçırabileceği ilgili dokümanları birden fazla perspektifle yakalama.

In [26]:
# ============================================
# RAG FUSION PIPELINE
# ============================================

QUERY_GEN_PROMPT = """Generate {n} different search queries that could help answer 
the following legal question. Each query should approach the question from a different 
angle or focus on different aspects. Return ONLY the queries, one per line.

Original question: {question}

Alternative queries:"""


def generate_query_variants(question: str, n: int = 4, model: str = None) -> list[str]:
    """Orijinal sorudan N farklı query üret."""
    response = call_llm(
        QUERY_GEN_PROMPT.format(question=question, n=n),
        model=model
    )
    
    queries = [question]  # Orijinal soruyu da dahil et
    for line in response.strip().split("\n"):
        line = re.sub(r'^[\d\.\-\)]+\s*', '', line.strip())
        if line and len(line) > 10:
            queries.append(line)
    
    return queries[:n+1]  # original + n variants


def reciprocal_rank_fusion(results_list: list[list], k: int = 60) -> list[dict]:
    """
    Reciprocal Rank Fusion (RRF).
    
    RRF_score(d) = Σ 1 / (k + rank_i(d))
    
    Birden fazla retrieval sonucunu tek bir ranked list'e birleştirir.
    """
    fused_scores = {}  # doc_text -> {score, metadata, text}
    
    for results in results_list:
        for rank, result in enumerate(results):
            doc_key = result["text"][:200]  # Dedup key
            
            if doc_key not in fused_scores:
                fused_scores[doc_key] = {
                    "text": result["text"],
                    "metadata": result["metadata"],
                    "rrf_score": 0,
                    "appearances": 0
                }
            
            fused_scores[doc_key]["rrf_score"] += 1.0 / (k + rank + 1)
            fused_scores[doc_key]["appearances"] += 1
    
    # Sort by RRF score
    ranked = sorted(fused_scores.values(), key=lambda x: x["rrf_score"], reverse=True)
    return ranked


def rag_fusion(
    question: str,
    collection,
    n_variants: int = 4,
    n_results: int = 5,
    top_k_fused: int = 5,
    model: str = None
) -> dict:
    """
    RAG Fusion Pipeline.
    
    Steps:
    1. Generate query variants
    2. Retrieve for each variant
    3. Apply Reciprocal Rank Fusion
    4. Generate answer from top-K fused results
    """
    import time
    start_time = time.time()
    
    # Step 1: Generate variants
    query_variants = generate_query_variants(question, n_variants, model)
    
    # Step 2: Retrieve for each variant
    all_retrieval_results = []
    
    for variant in query_variants:
        results = collection.query(
            query_texts=[variant],
            n_results=n_results
        )
        
        variant_results = []
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            variant_results.append({
                "text": doc,
                "metadata": meta,
                "similarity": 1 - dist
            })
        all_retrieval_results.append(variant_results)
    
    retrieval_time = time.time() - start_time
    
    # Step 3: RRF
    fused_results = reciprocal_rank_fusion(all_retrieval_results)
    top_results = fused_results[:top_k_fused]
    
    # Step 4: Generate answer
    context_parts = []
    sources = []
    
    for i, result in enumerate(top_results):
        source_info = f"[Source {i+1}: {result['metadata']['source_dataset']} | RRF={result['rrf_score']:.4f} | seen in {result['appearances']}/{len(query_variants)} queries]"
        context_parts.append(f"{source_info}\n{result['text']}")
        sources.append({
            "rank": i + 1,
            "dataset": result["metadata"]["source_dataset"],
            "domain": result["metadata"]["domain"],
            "rrf_score": result["rrf_score"],
            "appearances": result["appearances"],
            "text_preview": result["text"][:200]
        })
    
    context = "\n\n---\n\n".join(context_parts)
    prompt = BASIC_RAG_PROMPT.format(context=context, question=question)
    
    llm_start = time.time()
    answer = call_llm(prompt, model=model)
    llm_time = time.time() - llm_start
    
    total_time = time.time() - start_time
    
    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "query_variants": query_variants,
        "method": "rag_fusion",
        "model": model or LLM_MODEL,
        "metrics": {
            "retrieval_time": retrieval_time,
            "llm_time": llm_time,
            "total_time": total_time,
            "n_variants": len(query_variants),
            "n_sources": len(sources),
            "total_retrieved": sum(len(r) for r in all_retrieval_results),
            "unique_after_fusion": len(fused_results),
            "avg_rrf_score": np.mean([s["rrf_score"] for s in sources])
        }
    }


print(" RAG Fusion pipeline hazır.")

 RAG Fusion pipeline hazır.


In [27]:
# RAG Fusion test
fusion_test_q = "What legal protections exist against unfair contract termination in employment law?"

result = rag_fusion(fusion_test_q, collection)

print(" RAG Fusion Test")
print("=" * 80)
print(f"\n Orijinal Soru: {fusion_test_q}")
print(f"\n Üretilen Query Varyantları ({len(result['query_variants'])}):\n")
for i, qv in enumerate(result["query_variants"]):
    tag = "[original]" if i == 0 else f"[variant {i}]"
    print(f"  {tag} {qv}")

print(f"\n Cevap: {result['answer'][:400]}...")
print(f"\n Metrikler:")
print(f"  Toplam retrieval: {result['metrics']['total_retrieved']} doküman")
print(f"  Fusion sonrası unique: {result['metrics']['unique_after_fusion']}")
print(f"  Kullanılan top-K: {result['metrics']['n_sources']}")
print(f"  Toplam süre: {result['metrics']['total_time']:.2f}s")

print(f"\n Kaynaklar (RRF sıralı):")
for s in result["sources"]:
    print(f"  [{s['rank']}] {s['dataset']} | RRF: {s['rrf_score']:.4f} | {s['appearances']} query'de bulundu")

 RAG Fusion Test

 Orijinal Soru: What legal protections exist against unfair contract termination in employment law?

 Üretilen Query Varyantları (5):

  [original] What legal protections exist against unfair contract termination in employment law?
  [variant 1] "California wrongful termination laws - what constitutes an unfair reason for firing"
  [variant 2] "Legal remedies for breach of implied covenant of good faith and fair dealing in employment contracts"
  [variant 3] "Can an employer terminate an employee without cause under at-will employment states?"
  [variant 4] "What are the legal grounds for challenging a job termination as discriminatory or retaliatory?"

 Cevap: Under California employment law, employees have certain legal protections against unfair contract terminations. One key protection is the concept of "public policy." This means that an employer cannot terminate an employee for engaging in activities that align with or promote a clear mandate of public policy. T

---
## 7. LLM Karşılaştırması

3 model × 3 RAG yöntemi = 9 kombinasyon testi

| Model | Tip | Neden? |
|---|---|---|
| **LLama-8B-Lawyer** | Legal domain-specific | Hukuki veri ile eğitilmiş |
| **Mistral 7B Instruct** | General purpose | SaulLM'in base modeli, baseline |
| **Llama 3 8B Instruct** | General purpose | Farklı mimari ailesi |

In [28]:
# ============================================
# LLM KARŞILAŞTIRMA — SYSTEMATIC EVALUATION
# ============================================

MODELS = {
    "llama3_lawyer": "llama3-8b-lawyer-v2", 
    "mistral": "mistral-7b-instruct-v0.3",
    "llama3": "meta-llama-3-8b-instruct"
}

# Standardize edilmiş test soruları (kolay, orta, zor)
EVAL_QUESTIONS = [
    # Basit
    "What is the standard for determining if a contract clause is unconscionable?",
    # Orta
    "Under what circumstances can the right to privacy under Article 8 of ECHR be lawfully restricted?",
    # Zor
    "How do different legal systems approach the conflict between corporate liability clauses and consumer protection rights?"
]

RAG_METHODS = {
    "basic": basic_rag,
    "adaptive": adaptive_rag,
    "fusion": rag_fusion
}


def run_comparison(questions, models, methods, collection):
    """3x3 karşılaştırma matrisi çalıştır."""
    results = []
    
    for q_idx, question in enumerate(questions):
        for model_name, model_id in models.items():
            for method_name, method_fn in methods.items():
                print(f"  Q{q_idx+1} | {model_name} | {method_name}...", end=" ")
                
                try:
                    result = method_fn(question, collection, model=model_id)
                    results.append({
                        "question_id": q_idx + 1,
                        "question": question[:80],
                        "model": model_name,
                        "method": method_name,
                        "answer": result["answer"][:200],
                        "total_time": result["metrics"]["total_time"],
                        "n_sources": result["metrics"]["n_sources"],
                        "avg_similarity": result["metrics"].get("avg_similarity", 
                                          result["metrics"].get("avg_rrf_score", 0)),
                        "llm_calls": result["metrics"].get("llm_calls", 2),
                        "status": "success"
                    })
                    print(f"{result['metrics']['total_time']:.1f}s")
                except Exception as e:
                    results.append({
                        "question_id": q_idx + 1,
                        "question": question[:80],
                        "model": model_name,
                        "method": method_name,
                        "answer": f"ERROR: {str(e)}",
                        "total_time": 0,
                        "n_sources": 0,
                        "avg_similarity": 0,
                        "llm_calls": 0,
                        "status": "error"
                    })
                    print(f"{str(e)[:50]}")
    
    return pd.DataFrame(results)


print(" Karşılaştırma başlıyor...")
print(f"   {len(EVAL_QUESTIONS)} soru × {len(MODELS)} model × {len(RAG_METHODS)} yöntem = {len(EVAL_QUESTIONS)*len(MODELS)*len(RAG_METHODS)} test")
print("=" * 60)

comparison_df = run_comparison(EVAL_QUESTIONS, MODELS, RAG_METHODS, collection)

print(f"\n Karşılaştırma tamamlandı! {len(comparison_df)} test sonucu.")

 Karşılaştırma başlıyor...
   3 soru × 3 model × 3 yöntem = 27 test
  Q1 | llama3_lawyer | basic... 10.1s
  Q1 | llama3_lawyer | adaptive... 5.2s
  Q1 | llama3_lawyer | fusion... 15.5s
  Q1 | mistral | basic... 12.9s
  Q1 | mistral | adaptive... 10.3s
  Q1 | mistral | fusion... 29.7s
  Q1 | llama3 | basic... 20.3s
  Q1 | llama3 | adaptive... 11.3s
  Q1 | llama3 | fusion... 26.6s
  Q2 | llama3_lawyer | basic... 14.4s
  Q2 | llama3_lawyer | adaptive... 16.4s
  Q2 | llama3_lawyer | fusion... 22.9s
  Q2 | mistral | basic... 28.7s
  Q2 | mistral | adaptive... 15.0s
  Q2 | mistral | fusion... 33.8s
  Q2 | llama3 | basic... 28.5s
  Q2 | llama3 | adaptive... 19.7s
  Q2 | llama3 | fusion... 39.6s
  Q3 | llama3_lawyer | basic... 18.0s
  Q3 | llama3_lawyer | adaptive... 10.5s
  Q3 | llama3_lawyer | fusion... 22.9s
  Q3 | mistral | basic... 31.0s
  Q3 | mistral | adaptive... 99.3s
  Q3 | mistral | fusion... 32.9s
  Q3 | llama3 | basic... 29.8s
  Q3 | llama3 | adaptive... 118.0s
  Q3 | llama3 | fus

In [29]:
# Sonuç tablosu
print("KARŞILAŞTIRMA SONUÇLARI")
print("=" * 80)

# Model × Method ortalama süre
pivot_time = comparison_df.pivot_table(
    index="model", columns="method", values="total_time", aggfunc="mean"
).round(2)
print("\n Ortalama Yanıt Süresi (saniye):")
display(pivot_time) if hasattr(__builtins__, '__IPYTHON__') else print(pivot_time)

# Model × Method ortalama similarity
pivot_sim = comparison_df.pivot_table(
    index="model", columns="method", values="avg_similarity", aggfunc="mean"
).round(4)
print("\n Ortalama Retrieval Similarity:")
display(pivot_sim) if hasattr(__builtins__, '__IPYTHON__') else print(pivot_sim)

KARŞILAŞTIRMA SONUÇLARI

 Ortalama Yanıt Süresi (saniye):


method,adaptive,basic,fusion
model,,,
llama3,49.68,26.20,31.53
llama3_lawyer,10.68,14.16,20.42
mistral,41.53,24.21,32.16



 Ortalama Retrieval Similarity:


method,adaptive,basic,fusion
model,,,
llama3,0.5293,0.5245,0.0351
llama3_lawyer,0.5245,0.5245,0.0256
mistral,0.5335,0.5245,0.0405


In [31]:
# Visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ortalama Yanıt Süresi (s)", "Ortalama Kaynak Kalitesi")
)

model_colors = {"llama3_lawyer": "#E91E63", "mistral": "#2196F3", "llama3": "#4CAF50"}

for model_name in MODELS.keys():
    model_data = comparison_df[comparison_df["model"] == model_name]
    
    fig.add_trace(
        go.Bar(
            name=model_name,
            x=model_data.groupby("method")["total_time"].mean().index,
            y=model_data.groupby("method")["total_time"].mean().values,
            marker_color=model_colors[model_name]
        ),
        row=1, col=1
    )

fig.update_layout(
    title_text="LLM × RAG Method Karşılaştırması",
    height=450,
    barmode="group",
    template="plotly_white"
)
fig.show()

---
## 8. Değerlendirme & Sonuçlar

### 8.1 Karşılaştırma Özeti

In [32]:
# Final özet tablosu
print(" PROJE SONUÇ ÖZETİ")
print("=" * 80)

print(f"""
LexGLUE Legal RAG System — Sonuçlar

  Corpus:
   - {len(DATASETS)} dataset, {len(all_data)} CSV dosyası
   - Toplam {len(documents):,} orijinal belge
   - Chunking sonrası {len(chunked_documents):,} vektör kaydı
   - 6 hukuki alan: Case Law, Human Rights, EU Legislation, Contracts, Consumer Law, Supreme Court

  Mimari:
   - Embedding: {EMBEDDING_MODEL} (384 dim)
   - Vektör DB: ChromaDB (persistent, cosine similarity)
   - Chunking: Adaptive (threshold={CHUNK_THRESHOLD} kelime, size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})

  RAG Karşılaştırması:
   - Basic RAG: Hızlı, basit sorular için yeterli
   - Adaptive RAG: Complexity routing ile verimli kaynak kullanımı
   - RAG Fusion: Multi-query + RRF ile en kapsamlı retrieval

  LLM Karşılaştırması:
   - Llama 3 - Lawyer (GGUF): Domain-specific, hukuki terminoloji avantajı
   - Mistral 7B: İyi genel performans, baseline
   - Llama 3 8B: Güncel training, güçlü reasoning
""")

 PROJE SONUÇ ÖZETİ

LexGLUE Legal RAG System — Sonuçlar

  Corpus:
   - 7 dataset, 21 CSV dosyası
   - Toplam 3,500 orijinal belge
   - Chunking sonrası 37,970 vektör kaydı
   - 6 hukuki alan: Case Law, Human Rights, EU Legislation, Contracts, Consumer Law, Supreme Court

  Mimari:
   - Embedding: all-MiniLM-L6-v2 (384 dim)
   - Vektör DB: ChromaDB (persistent, cosine similarity)
   - Chunking: Adaptive (threshold=512 kelime, size=1000, overlap=150)

  RAG Karşılaştırması:
   - Basic RAG: Hızlı, basit sorular için yeterli
   - Adaptive RAG: Complexity routing ile verimli kaynak kullanımı
   - RAG Fusion: Multi-query + RRF ile en kapsamlı retrieval

  LLM Karşılaştırması:
   - Llama 3 - Lawyer (GGUF): Domain-specific, hukuki terminoloji avantajı
   - Mistral 7B: İyi genel performans, baseline
   - Llama 3 8B: Güncel training, güçlü reasoning



### 8.2 İleri Vizyon

Bu proje bir başlangıç noktasıdır. İleri adımlar:

| # | İyileştirme | Beklenen Etki |
|---|---|---|
| 1 | **Legal-domain embeddings** (`nlpaueb/legal-bert-base-uncased`) | Retrieval precision +15-20% |
| 2 | **Hybrid Search** (Semantic + BM25 keyword) | Exact match + semantic anlam birleşimi |
| 3 | **Cross-encoder Re-ranking** | Top-K sonuçların kalite sıralaması |
| 4 | **RAGAS evaluation framework** | Otomatik faithfulness, relevance, hallucination ölçümü |
| 5 | **Multi-lingual support** | Türkçe hukuki belgeler için genişleme |
| 6 | **Fine-tuning** | LexGLUE üzerinde task-specific fine-tuning |
| 7 | **Agentic RAG** | LangGraph ile multi-step reasoning agent |

---

## Kaynaklar

- **Dataset:** [LexGLUE - Kaggle](https://www.kaggle.com/datasets/thedevastator/lexglue-legal-nlp-benchmark-dataset)
- **LexGLUE Paper:** Chalkidis et al., "LexGLUE: A Benchmark Dataset for Legal Language Understanding in English", ACL 2022
- **RAG Fusion:** Raudaschl, "Forget RAG, the Future is RAG-Fusion", 2023
- **Adaptive RAG:** Jeong et al., "Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity", 2024
- **LegalBench:** Guha et al., "LegalBench: A Collaboratively Built Benchmark for Measuring Legal Reasoning in LLMs", 2023